# 01 — Transfer learning with ResNet50 (TensorFlow/Keras)

TensorFlow-only lesson. Dataset contract: the same Homer/Bart folder layout as lesson 00, or any two-class image-folder dataset.

## 1. Version and dataset setup

In [ ]:
import tensorflow as tf
from pathlib import Path
print('TensorFlow',tf.__version__,'Keras',tf.keras.__version__)
ROOT=Path('data/homer_bart_2'); TRAIN=ROOT/'training_set'; TEST=ROOT/'test_set'
assert TRAIN.exists() and TEST.exists()

## 2. Load images and preserve pretrained preprocessing

ResNet50 expects its own preprocessing. `preprocess_input` converts RGB inputs to the channel convention and scale used by ImageNet training; do not also divide by 255 unless the model's preprocessing contract explicitly requires it.

In [ ]:
SIZE=(224,224); BATCH=24; SEED=42
train=tf.keras.utils.image_dataset_from_directory(TRAIN,image_size=SIZE,batch_size=BATCH,validation_split=.2,subset='training',seed=SEED,label_mode='binary')
valid=tf.keras.utils.image_dataset_from_directory(TRAIN,image_size=SIZE,batch_size=BATCH,validation_split=.2,subset='validation',seed=SEED,label_mode='binary',shuffle=False)
test=tf.keras.utils.image_dataset_from_directory(TEST,image_size=SIZE,batch_size=BATCH,label_mode='binary',shuffle=False)
labels=train.class_names
prep=tf.keras.applications.resnet50.preprocess_input
augment=tf.keras.Sequential([tf.keras.layers.RandomFlip('horizontal'),tf.keras.layers.RandomRotation(.03)])
train=train.map(lambda x,y:(prep(augment(x,training=True)),y)).prefetch(tf.data.AUTOTUNE)
valid=valid.map(lambda x,y:(prep(x),y)).cache().prefetch(tf.data.AUTOTUNE)
test=test.map(lambda x,y:(prep(x),y)).cache().prefetch(tf.data.AUTOTUNE)

## 3. Frozen feature extractor

In [ ]:
base=tf.keras.applications.ResNet50(include_top=False,weights='imagenet',input_shape=(*SIZE,3))
base.trainable=False
inputs=tf.keras.Input((*SIZE,3)); x=base(inputs,training=False); x=tf.keras.layers.GlobalAveragePooling2D()(x); x=tf.keras.layers.Dropout(.25)(x); outputs=tf.keras.layers.Dense(1,activation='sigmoid')(x)
model=tf.keras.Model(inputs,outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),loss='binary_crossentropy',metrics=['accuracy',tf.keras.metrics.AUC(name='auc')])
model.summary()

## 4. Train classifier head

In [ ]:
cb=[tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=4,restore_best_weights=True),tf.keras.callbacks.ModelCheckpoint('resnet50_frozen.keras',save_best_only=True,monitor='val_loss')]
h1=model.fit(train,validation_data=valid,epochs=15,callbacks=cb)

## 5. Fine-tune the final ResNet stage

Unfreeze only the final convolutional stage, keep BatchNorm layers frozen, and recompile with a much smaller learning rate. Recompilation is required after changing trainable flags.

In [ ]:
base.trainable=True
for layer in base.layers:
    layer.trainable=layer.name.startswith('conv5_') and not isinstance(layer,tf.keras.layers.BatchNormalization)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),loss='binary_crossentropy',metrics=['accuracy',tf.keras.metrics.AUC(name='auc')])
h2=model.fit(train,validation_data=valid,epochs=10,callbacks=cb)
model.save('resnet50_finetuned.keras')

## 6. Final test report

In [ ]:
from sklearn.metrics import classification_report
import numpy as np
best=tf.keras.models.load_model('resnet50_frozen.keras')
y=np.concatenate([v.numpy().ravel() for _,v in test]).astype(int); p=best.predict(test,verbose=0).ravel()>=.5
print(classification_report(y,p,target_names=labels,digits=4))

## Practical tips

- Record the exact ImageNet weights identifier and preprocessing.
- Use a validation set for selecting the unfreezing depth and learning rate.
- Avoid training the whole backbone from the start on a small dataset.
- Evaluate class-wise errors and inspect false positives/negatives.
- For imbalanced data, consider class weights computed from training labels only.